In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
#for dirname, _, filenames in os.walk('/kaggle/input'):
 #   for filename in filenames:
       # print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
#!/usr/bin/env python3
"""RSNA Knee Abnormality Detection -- PHASE 2: end-to-end training (backbone + transformer head)
with OPTIONAL series-shared augmentation.

Phase 1 trained only the transformer head on cached features of a frozen backbone. Phase 2 unfreezes
the backbone and trains everything together, warm-starting from the phase-1 weights:

    backbone  <- Config.backbone_ckpt   (the same fine-tuned backbone phase 1 used)
    head      <- Config.head_ckpt       (knee_head_*.pt written by phase 1; optional but recommended)

Resume / continue training: set Config.resume_ckpt (or --resume path) to a knee_<tag>.pt written by this script;
it loads backbone + head weights and starts a new run tagged <tag>_resume.

Series-shared augmentation (Config.aug = True/False, or --aug 1 / --aug 0 on the command line)
    Every window of one SERIES (= one slot of one study) receives the SAME random draw of rotation,
    scale, shift, gamma and gain, so the geometry stays consistent across the slices of a series
    (the 3 channels of each 2.5D window and all windows of the slot move together). Different slots
    of the same study get independent draws. Augmentation runs on the GPU, only while training.
    With aug off, training sees the unmodified windows (random window subset per study only), which
    makes a clean A/B comparison. The run tag is chosen automatically (e2e_aug / e2e_noaug) so the two
    runs never overwrite each other.

Changes in this version
    * single GPU by default (multi_gpu = False)
    * tqdm progress bars for training and for every evaluation, plus a plain-text s/it print
      every PROGRESS_EVERY iterations (reliable in "Save & Run All" logs)
    * torch.backends.cudnn.benchmark = False (eval uses a different number of windows per study,
      so autotune would re-benchmark the whole network on every new shape)

Sections
  0. Config / CLI                5. Series-shared augmentation (GPU)
  1. Label preparation           6. Model (backbone + KneeHeadTF)
  2. Slot layout + window meta   7. Train / evaluate
  3. Corpus loading              8. Post-training (OOF, SWA, epoch ensemble)
  4. Dataset + label sets        9. main()
"""
import os
import re
import glob
import json
import math
import time
import random
import argparse
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm
import timm


# ==============================================================================
# 0. CONFIG / CLI
# ==============================================================================
LAB = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA", "PF OA",
       "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]

PROGRESS_EVERY = 50             # plain-text progress line every N micro-batches


class Config:
    # ---- paths -------------------------------------------------------------
    HERE = "/kaggle/working"
    comp_train_csv = "/kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv"
    corpus_root = "/kaggle/input/datasets/chikonzeroselemani/rsna-corpus/corpus_experiment"
    backbone_ckpt = ("/kaggle/input/datasets/chikonzeroselemani/coatnet-fature-extractor/"
                     "convnext_tiny_knee_backbone.pt")
    head_ckpt = None            # path to knee_head_<tag>.pt from phase 1 (recommended). None -> random head
    labels_csv = "/kaggle/input/datasets/chikonzeroselemani/rsna-frozen/balas_labels.csv"           # path to an existing new_labels.csv; None -> rebuild it from the sources below

    # ---- RESUME / CONTINUE TRAINING ----------------------------------------
    # Path to a checkpoint written by THIS script: knee_<tag>.pt (best epoch) or knee_<tag>_swa.pt.
    # It holds backbone + head weights, so when set it REPLACES backbone_ckpt / head_ckpt entirely.
    # Weights only (no optimizer state): a fresh OneCycle schedule starts, so consider lower LRs, e.g.
    # bb_lr 1e-5 / head_lr 1e-4. The new run gets its own tag ("<tag>_resume") so it never overwrites the old one.
    # Examples:
    #   resume_ckpt = "/kaggle/working/knee_e2e_aug.pt"                       (same session)
    #   resume_ckpt = "/kaggle/input/datasets/<you>/<dataset>/knee_e2e_aug.pt" (saved as a Kaggle dataset)
    # or on the command line:  --resume /path/to/knee_e2e_aug.pt
    # resume_ckpt = "/kaggle/input/datasets/chikonzeroselemani/convnext-train/knee_e2e_aug_resume.pt"
    resume_ckpt = None
    
    # ---- label sources (only used when labels_csv is None) ------------------
    # oof_pseudolabels = "/kaggle/input/datasets/chikonzeroselemani/pseudolabelsnew/pseudoLabelsNew.txt"
    # pseudolabel_files = [
    #     "/kaggle/input/datasets/chikonzeroselemani/rsna-pseudolabels/train_pseudolabels.csv",
    #     "/kaggle/input/datasets/chikonzeroselemani/rsnapseudolabel2/RSNApseudolabels2.txt",
    #     "/kaggle/input/datasets/chikonzeroselemani/rsnapseudolabels3/RSNApseudolabels3.txt",
    #     "/kaggle/input/datasets/chikonzeroselemani/rsnapseudolabels4/RSNApseudolabels4.txt",
    # ]
    llm_weight = 0.5

    # ---- model -------------------------------------------------------------
    arch = "convnext_tiny.in12k_ft_in1k"
    res = 224
    norm = "imagenet"               # "none" | "imagenet"  (must match how the backbone was trained)
    drop_path = 0.0             # stochastic depth for the backbone (0 = off; 0.1 is a common fine-tune value)
    freeze_stages = 0           # freeze stem + first N backbone stages (saves memory/time)
    grad_ckpt = True            # gradient checkpointing in the backbone (slower, much less memory)
    tf_dim = 256
    tf_depth = 2
    tf_heads = 8
    tf_drop = 0.1
    head_drop = 0.2

    # ---- training ----------------------------------------------------------
    epochs = 12
    multi_gpu = False           # GPU 0 only (True -> nn.DataParallel over every visible GPU)
    bs_per_gpu = 4              # studies per GPU per micro-batch (memory is set by this, not by the GPU count)
    target_batch = 8            # studies per optimizer step; gradient accumulation is derived from it
    bs = None                   # (derived) total studies per micro-batch  = bs_per_gpu * n_gpu
    accum_steps = None          # (derived) = max(1, round(target_batch / bs))
    n_gpu = 1                   # (derived) number of GPUs actually used
    k_train = 24                # random windows per study at train time (eval uses ALL windows)
    bb_lr = 2e-4
    head_lr = 2e-3              # lowered because the head is warm-started; use ~5e-4 for a fresh head
    wd = 0.02
    pct_start = 0.1
    clip_grad = 3.0
    topk = 3
    workers = 4
    amp = True                  # fp16 autocast for the backbone (T4 has no bf16 conv engine)

    # ---- series-shared augmentation ----------------------------------------
    aug = True                  # <-- main switch (or --aug 0/1)
    aug_p = 1.0                 # probability that a given series is augmented at all
    aug_rot = 12.0              # max rotation, degrees  (uniform in +-aug_rot)
    aug_scale = 0.10            # scale in 1 +- aug_scale
    aug_shift = 0.06            # translation in +-aug_shift (fraction of half-width)
    aug_gamma = 0.25            # gamma = exp(+-aug_gamma)
    aug_gain = 0.10             # gain in 1 +- aug_gain

    # ---- misc --------------------------------------------------------------
    tag = None                  # None -> "e2e_aug" / "e2e_noaug"
    seed = 42
    smoke = False
    limit = 0                   # >0 -> only the first `limit` training studies
    eval_baseline = True        # evaluate the warm-started model once before training (sanity check)

    # ---- cross-validation (optional) ---------------------------------------
    folds = 0
    fold = -1
    fold_file = None            # json: {"folds": {study_uid: fold_index}}


def apply_cli_overrides(cfg):
    """Tiny CLI so the aug on/off comparison needs no code edit. Unknown args (Jupyter) are ignored."""
    p = argparse.ArgumentParser(add_help=False)
    p.add_argument("--aug", type=int, choices=[0, 1])
    p.add_argument("--tag", type=str)
    p.add_argument("--epochs", type=int)
    p.add_argument("--smoke", action="store_true")
    p.add_argument("--resume", type=str)
    p.add_argument("--seed", type=int)
    p.add_argument("--bb_lr", type=float)
    p.add_argument("--head_lr", type=float)
    a, _ = p.parse_known_args()
    if a.aug is not None:
        cfg.aug = bool(a.aug)
    if a.tag:
        cfg.tag = a.tag
    if a.epochs:
        cfg.epochs = a.epochs
    if a.resume:
        cfg.resume_ckpt = a.resume
    if a.seed is not None:
        cfg.seed = a.seed
    if a.bb_lr:
        cfg.bb_lr = a.bb_lr
    if a.head_lr:
        cfg.head_lr = a.head_lr
    if a.smoke:
        cfg.smoke = True
    if cfg.smoke:
        cfg.epochs, cfg.limit, cfg.workers, cfg.k_train = 2, 24, 0, 8
    if cfg.tag is None:
        cfg.tag = "e2e_aug" if cfg.aug else "e2e_noaug"
        if cfg.resume_ckpt:
            cfg.tag += "_resume"
    if cfg.resume_ckpt:
        cfg.eval_baseline = True        # always re-evaluate the loaded weights: confirms the checkpoint loaded
    return cfg


def setup_gpus(cfg):
    """Decide how many GPUs to use and derive the batch size / gradient accumulation from it.

    With nn.DataParallel the micro-batch of `bs` studies is split evenly across the GPUs, so each GPU
    sees `bs_per_gpu` studies: per-GPU memory stays the same while throughput roughly doubles on 2 GPUs.
    The optimizer-step batch stays ~target_batch studies, so learning rates keep their meaning."""
    avail = torch.cuda.device_count()
    cfg.n_gpu = max(1, avail) if cfg.multi_gpu else 1
    cfg.bs = cfg.bs_per_gpu * cfg.n_gpu
    cfg.accum_steps = max(1, round(cfg.target_batch / cfg.bs))
    names = ", ".join(torch.cuda.get_device_name(i) for i in range(min(cfg.n_gpu, avail))) or "cpu"
    print(f"[gpu] using {cfg.n_gpu} of {avail} GPU(s): {names} | micro-batch {cfg.bs} studies "
          f"({cfg.bs_per_gpu}/GPU) x accum {cfg.accum_steps} = {cfg.bs * cfg.accum_steps} studies per step",
          flush=True)
    return cfg


# ==============================================================================
# 1. LABEL PREPARATION
# ==============================================================================
def prepare_labels(cfg):
    """Blend LLM soft labels with pseudo-labels, append gold rows, write new_labels.csv."""
    soft = pd.read_csv(cfg.oof_pseudolabels)
    soft["StudyInstanceUID"] = soft["StudyInstanceUID"].astype(str)

    pseudo = pd.concat([pd.read_csv(p) for p in cfg.pseudolabel_files], ignore_index=True)
    pseudo["StudyInstanceUID"] = pseudo["StudyInstanceUID"].astype(str)
    pseudo = pseudo.drop_duplicates("StudyInstanceUID", keep="last").set_index("StudyInstanceUID")

    w_llm, w_ps = cfg.llm_weight, 1.0 - cfg.llm_weight
    print(f"[labels] LLM_WEIGHT {w_llm} | PSEUDO_WEIGHT {w_ps}", flush=True)
    common = soft["StudyInstanceUID"].isin(pseudo.index)
    for col in LAB:
        llm_v = soft.loc[common, col].astype(float)
        ps_v = soft.loc[common, "StudyInstanceUID"].map(pseudo[col]).astype(float)
        soft.loc[common, col] = w_llm * llm_v + w_ps * ps_v
    print(f"[labels] blended {int(common.sum())} / {len(soft)} studies", flush=True)

    tr = pd.read_csv(cfg.comp_train_csv)
    tr["StudyInstanceUID"] = tr["StudyInstanceUID"].astype(str)
    gold_df = tr[tr[LAB].notna().all(axis=1)].copy()

    cols = ["StudyInstanceUID"] + LAB
    soft = pd.concat([soft[cols], gold_df[cols]], ignore_index=True).drop_duplicates("StudyInstanceUID", keep="last")
    out = os.path.join(cfg.HERE, "new_labels.csv")
    soft.to_csv(out, index=False)
    print(f"[labels] wrote {out} ({len(soft)} rows)", flush=True)
    return out


# ==============================================================================
# 2. SLOT LAYOUT + WINDOW METADATA  (identical to phase 1)
# ==============================================================================
SLOTS = [("sag", 0, 22), ("sag", 22, 40), ("cor", 40, 55), ("cor", 55, 65), ("ax", 65, 80)]
N_SLICES = 80
N_SLOTS = len(SLOTS)
WIN = 3
WIN_STRIDE = 1


def build_windows(slots=SLOTS, win=WIN, stride=WIN_STRIDE):
    """(win_idx [n_win, win], win_slot [n_win], win_pos [n_win] = position in own slot, 0..1)."""
    idx, slot_of, pos = [], [], []
    for s, (plane, lo, hi) in enumerate(slots):
        assert hi - lo >= win, f"slot {s} ({plane} {lo}:{hi}) shorter than window size {win}"
        starts = list(range(lo, hi - win + 1, stride))
        span = max(len(starts) - 1, 1)
        for i, start in enumerate(starts):
            idx.append(list(range(start, start + win)))
            slot_of.append(s)
            pos.append(i / span)
    return (np.asarray(idx, dtype=np.int64), np.asarray(slot_of, dtype=np.int64),
            np.asarray(pos, dtype=np.float32))


WIN_IDX, WIN_SLOT, WIN_POS = build_windows()
MAX_C = len(WIN_IDX)
for _w, _s in zip(WIN_IDX, WIN_SLOT):
    assert SLOTS[_s][1] <= _w.min() and _w.max() < SLOTS[_s][2], "window leaks out of its slot"
print(f"[windows] {MAX_C} windows | per slot: "
      + ", ".join(f"{p}[{lo}:{hi}]={n}" for (p, lo, hi), n in zip(SLOTS, np.bincount(WIN_SLOT))), flush=True)


# ==============================================================================
# 3. CORPUS LOADING  (identical to phase 1)
# ==============================================================================
_NPY = re.compile(r"^(?P<prefix>.*?)_?(?P<kind>ids|vols|masks)\.npy$")


def _discover_parts(root):
    groups = {}
    found = sorted(glob.glob(os.path.join(str(root), "**", "*.npy"), recursive=True))
    for f in found:
        m = _NPY.match(os.path.basename(f))
        if m:
            groups.setdefault((os.path.dirname(f), m["prefix"]), {})[m["kind"]] = f
    parts = [g for _, g in sorted(groups.items()) if "ids" in g and "vols" in g]
    if not parts:
        raise FileNotFoundError(f"No <prefix>_ids.npy + <prefix>_vols.npy pair under {root}.")
    return parts


class MultiPartVols:
    """Several memory-mapped (n_i, N_SLICES, H, W) arrays presented as one array."""

    def __init__(self, parts):
        self.parts = parts
        self.offsets = np.cumsum([0] + [p.shape[0] for p in parts])
        self.shape = (int(self.offsets[-1]),) + tuple(parts[0].shape[1:])

    def __len__(self):
        return self.shape[0]

    def __getitem__(self, row):
        j = int(np.searchsorted(self.offsets, row, side="right") - 1)
        return self.parts[j][row - self.offsets[j]]


def open_ids(root):
    return np.concatenate([np.load(g["ids"], allow_pickle=True).astype(str) for g in _discover_parts(root)])


def open_corpus(root):
    parts = _discover_parts(root)
    vols_parts = []
    for g in parts:
        v = np.load(g["vols"], mmap_mode="r")
        n_ids = np.load(g["ids"], allow_pickle=True).shape[0]
        assert v.ndim == 4 and v.shape[1] == N_SLICES, \
            f"{os.path.basename(g['vols'])}: expected (N, {N_SLICES}, H, W), got {v.shape}"
        assert v.shape[0] == n_ids, f"{os.path.basename(g['vols'])}: {v.shape[0]} volumes but {n_ids} ids"
        vols_parts.append(v)
    vols = MultiPartVols(vols_parts)
    masks = None
    if all("masks" in g for g in parts):
        masks = np.concatenate([np.load(g["masks"]) for g in parts], axis=0)
        if masks.shape != (vols.shape[0], N_SLICES):
            print(f"[corpus] WARNING: masks {masks.shape} != {(vols.shape[0], N_SLICES)} - ignoring masks", flush=True)
            masks = None
    return vols, masks


def candidate_windows(vol, mask_row):
    """Ordered ids (rows of WIN_IDX) of every slot-safe window whose 3 slices are all non-empty."""
    if mask_row is not None:
        valid = mask_row > 0
    else:
        valid = np.asarray(vol).reshape(N_SLICES, -1).any(axis=1)
    cs = np.flatnonzero(valid[WIN_IDX].all(axis=1))
    if len(cs) == 0:
        cs = np.arange(len(WIN_IDX))
    return cs


# ==============================================================================
# 4. DATASET + LABEL SETS
# ==============================================================================
class StudyWindowsE2E(Dataset):
    """Per-study bag of 2.5D windows (3 consecutive slices of ONE slot as RGB), full image resized to `res`.

    train=True : k random DISTINCT windows (sorted); studies with fewer than k get masked padding.
    train=False: ALL windows of the study (use batch_size=1).
    Returns (x [K,3,res,res] fp16 in 0..1, slot [K] long, pos [K] float, pad [K] bool, y [12]).
    No augmentation happens here -- it is applied on the GPU (section 5) so it can be shared per series."""

    def __init__(self, corpus_root, ids, id2row, labels, res, k, train, norm="none"):
        self.corpus_root, self.ids, self.id2row, self.labels = corpus_root, ids, id2row, labels
        self.res, self.k, self.train, self.norm = res, k, train, norm
        self._mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        self._std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        self.vols = None
        self.masks = None

    def __len__(self):
        return len(self.ids)

    def _ensure(self):
        if self.vols is None:
            self.vols, self.masks = open_corpus(self.corpus_root)

    def _resize(self, tri):
        t = torch.from_numpy(np.ascontiguousarray(tri))
        if t.shape[-1] != self.res or t.shape[-2] != self.res:
            t = F.interpolate(t[None], size=(self.res, self.res), mode="bilinear", align_corners=False)[0]
        return t

    def __getitem__(self, i):
        self._ensure()
        uid = self.ids[i]
        row = self.id2row[uid]
        vol = self.vols[row]
        cs = candidate_windows(vol, None if self.masks is None else self.masks[row])

        if self.train:
            if len(cs) > self.k:
                cs = np.array(sorted(random.sample(list(cs), self.k)), dtype=np.int64)
            K = self.k
        else:
            K = len(cs)
        n = len(cs)

        x = torch.zeros(K, 3, self.res, self.res, dtype=torch.float16)
        for j, w in enumerate(cs):
            tri = np.asarray(vol[WIN_IDX[w]]).astype(np.float32)
            if vol.dtype == np.uint8:
                tri /= 255.0
            t = self._resize(tri)
            if self.norm == "imagenet":
                t = (t - self._mean) / self._std
            x[j] = t
        slot = torch.zeros(K, dtype=torch.long)
        pos = torch.zeros(K, dtype=torch.float32)
        slot[:n] = torch.from_numpy(WIN_SLOT[cs])
        pos[:n] = torch.from_numpy(WIN_POS[cs])
        pad = torch.arange(K) >= n
        y = torch.from_numpy(self.labels[uid])
        return x, slot, pos, pad, y


def build_label_sets(cfg, ids, labels_csv):
    idset = set(ids)
    tr = pd.read_csv(cfg.comp_train_csv)
    tr["StudyInstanceUID"] = tr["StudyInstanceUID"].astype(str)
    gold_df = tr[tr[LAB].notna().all(axis=1)].copy().set_index("StudyInstanceUID")
    gold_ids = [u for u in gold_df.index if u in idset]
    print(f"[data] gold-labeled studies present in corpus: {len(gold_ids)}", flush=True)

    soft = pd.read_csv(labels_csv)
    soft["StudyInstanceUID"] = soft["StudyInstanceUID"].astype(str)
    soft = soft.set_index("StudyInstanceUID")

    goldset = set(gold_ids)
    train_ids = [u for u in ids if u in soft.index and u not in goldset]

    oof_ids = []
    if cfg.folds > 0:
        assert 0 <= cfg.fold < cfg.folds, f"fold must be in [0,{cfg.folds}) when folds>0"
        assert cfg.fold_file and os.path.exists(cfg.fold_file), "fold_file required"
        with open(cfg.fold_file, "r") as f:
            fmap = json.load(f)["folds"]
        held = {u for u in train_ids if fmap.get(u, -1) == cfg.fold}
        oof_ids = [u for u in train_ids if u in held]
        train_ids = [u for u in train_ids if u not in held]
        print(f"[cv] fold {cfg.fold}/{cfg.folds}: train {len(train_ids)} | OOF {len(oof_ids)}", flush=True)

    if cfg.limit:
        train_ids = train_ids[:cfg.limit]

    labels = {u: soft.loc[u, LAB].values.astype(np.float32) for u in train_ids}
    for u in oof_ids:
        labels[u] = soft.loc[u, LAB].values.astype(np.float32)
    for u in gold_ids:
        labels[u] = gold_df.loc[u, LAB].values.astype(np.float32)
    print(f"[data] train {len(train_ids)} | gold-val {len(gold_ids)}"
          + (f" | oof {len(oof_ids)}" if oof_ids else ""), flush=True)

    prev = np.clip(np.stack([labels[u] for u in train_ids]).mean(0), 0.03, 0.7)
    pos_weight = np.clip((1 - prev) / prev, 1, 10).astype(np.float32)
    return SimpleNamespace(train_ids=train_ids, gold_ids=gold_ids, oof_ids=oof_ids,
                           labels=labels, pos_weight=pos_weight)


# ==============================================================================
# 5. SERIES-SHARED AUGMENTATION (GPU)
# ==============================================================================
@torch.no_grad()
def series_shared_augment(x, slot, cfg):
    """x [B,K,3,H,W] float in 0..1 (on GPU) | slot [B,K] long -> augmented x, same shape.

    One random draw of (rotation, scale, shift x/y, gamma, gain) per SERIES = per (study, slot).
    Every window of that series -- and all 3 channels inside each window -- gets the identical
    affine warp and intensity change, so the stacked slices stay geometrically consistent.
    Slots of the same study draw independently. Padded windows are all zeros and stay zeros."""
    B, K = slot.shape
    dev = x.device

    def u():                                           # uniform in [-1, 1], one value per (study, slot)
        return torch.rand(B, N_SLOTS, device=dev) * 2 - 1

    on = (torch.rand(B, N_SLOTS, device=dev) < cfg.aug_p).float()   # series left untouched when 0
    angle = u() * math.radians(cfg.aug_rot) * on
    scale = 1 + u() * cfg.aug_scale * on
    tx = u() * cfg.aug_shift * on
    ty = u() * cfg.aug_shift * on
    gamma = torch.exp(u() * cfg.aug_gamma * on)
    gain = 1 + u() * cfg.aug_gain * on

    def per_window(p):                                 # [B, N_SLOTS] -> [B*K], each window reads its series' value
        return p.gather(1, slot).reshape(-1)

    angle, scale, tx, ty, gamma, gain = map(per_window, (angle, scale, tx, ty, gamma, gain))
    cos, sin = torch.cos(angle) * scale, torch.sin(angle) * scale
    theta = torch.stack([torch.stack([cos, -sin, tx], 1), torch.stack([sin, cos, ty], 1)], 1)   # [B*K,2,3]

    xf = x.flatten(0, 1)                                                                          # [B*K,3,H,W]
    grid = F.affine_grid(theta, tuple(xf.shape), align_corners=False)
    xf = F.grid_sample(xf, grid, mode="bilinear", padding_mode="zeros", align_corners=False)
    xf = xf.clamp(0, 1).pow(gamma.view(-1, 1, 1, 1)) * gain.view(-1, 1, 1, 1)
    return xf.clamp(0, 1).view_as(x)


# ==============================================================================
# 6. MODEL
# ==============================================================================
def build_backbone(cfg, pretrained=False):
    is_vit = any(k in cfg.arch for k in ("vit", "deit", "dinov2", "eva", "beit"))
    kw = dict(pretrained=pretrained, num_classes=0, in_chans=3)
    if is_vit:
        kw.update(global_pool="token", dynamic_img_size=True)
    else:
        kw.update(global_pool="avg")
    if cfg.drop_path > 0:
        kw["drop_path_rate"] = cfg.drop_path
    return timm.create_model(cfg.arch, **kw)


def load_backbone_weights(cfg, bb, ckpt_path):
    """Accepts a feature-extractor file ({'backbone': bare sd, ...}) or a classifier checkpoint ('backbone.*')."""
    ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    meta = {}
    if isinstance(ck, dict) and isinstance(ck.get("backbone"), dict) \
            and not any(k.startswith("backbone.") for k in ck):
        bbst = ck["backbone"]
        meta = {k: ck.get(k) for k in ("arch", "res", "feature_dim", "source_epoch", "source_ckpt")}
        if meta.get("arch") and meta["arch"] != cfg.arch:
            print(f"[ckpt] WARNING: file arch '{meta['arch']}' != cfg.arch '{cfg.arch}'", flush=True)
        if meta.get("res") and meta["res"] != cfg.res:
            print(f"[ckpt] WARNING: file res {meta['res']} != cfg.res {cfg.res}", flush=True)
    else:
        st = ck.get("student", ck.get("model", ck)) if isinstance(ck, dict) else ck
        bbst = {k[len("backbone."):]: v for k, v in st.items() if k.startswith("backbone.")}
        if not bbst:
            bbst = {k: v for k, v in st.items() if isinstance(v, torch.Tensor)}
        meta["source_epoch"] = ck.get("epoch", "?") if isinstance(ck, dict) else "?"
    assert bbst, f"no backbone tensors found in {ckpt_path}"
    missing, unexpected = bb.load_state_dict(bbst, strict=False)
    assert not missing, f"backbone tensors missing from checkpoint: {missing[:5]} ..."
    src = f"{meta.get('source_ckpt') or os.path.basename(ckpt_path)}@ep{meta.get('source_epoch', '?')}"
    print(f"[ckpt] backbone <- {src} | {len(bbst)} tensors, unexpected {len(unexpected)}", flush=True)
    return src


class KneeHeadTF(nn.Module):
    """window features -> Dropout/Linear/LayerNorm -> + slot emb + slice-position emb
       -> transformer over ALL windows of the study -> per-finding attention pooling -> 12 logits."""

    def __init__(self, F_dim, n=12, dim=256, depth=2, heads=8, n_slots=5, drop=0.2, tf_drop=0.1):
        super().__init__()
        self.proj = nn.Sequential(nn.Dropout(drop), nn.Linear(F_dim, dim), nn.LayerNorm(dim))
        self.slot_emb = nn.Embedding(n_slots, dim)
        self.pos_emb = nn.Sequential(nn.Linear(1, dim), nn.GELU(), nn.Linear(dim, dim))
        layer = nn.TransformerEncoderLayer(dim, heads, dim * 2, dropout=tf_drop, activation="gelu",
                                           batch_first=True, norm_first=True)
        self.tf = nn.TransformerEncoder(layer, depth, enable_nested_tensor=False) if depth else None
        self.norm = nn.LayerNorm(dim)
        self.att = nn.Sequential(nn.Linear(dim, 256), nn.Tanh(), nn.Dropout(drop), nn.Linear(256, n))
        self.clsW = nn.Parameter(torch.zeros(n, dim))
        self.clsb = nn.Parameter(torch.zeros(n))
        nn.init.trunc_normal_(self.clsW, std=0.02)

    def forward(self, feats, slot, pos, pad):
        h = self.proj(feats) + self.slot_emb(slot) + self.pos_emb(pos.unsqueeze(-1))
        if self.tf is not None:
            h = self.tf(h, src_key_padding_mask=pad)
        h = self.norm(h)
        a = self.att(h).masked_fill(pad.unsqueeze(-1), -1e4).softmax(1)
        pooled = torch.einsum("bkn,bkd->bnd", a, h)
        return (pooled * self.clsW).sum(-1) + self.clsb


class KneeModelE2E(nn.Module):
    """backbone (fp16 autocast, trainable) + KneeHeadTF (always fp32). State-dict keys: backbone.* / head.*"""

    def __init__(self, backbone, head, amp=True):
        super().__init__()
        self.backbone = backbone
        self.head = head
        self.amp = amp

    def forward(self, x, slot, pos, pad):
        """x [B,K,3,H,W] float | slot, pos, pad [B,K] -> logits [B,12]."""
        B, K = x.shape[:2]
        with torch.autocast("cuda", dtype=torch.float16, enabled=self.amp and x.is_cuda):
            f = self.backbone(x.flatten(0, 1))
        return self.head(f.float().view(B, K, -1), slot, pos, pad)


def build_model(cfg, dev):
    bb = build_backbone(cfg, pretrained=False)
    if cfg.resume_ckpt:
        bb_src = f"resume:{os.path.basename(cfg.resume_ckpt)}"      # weights come from the resume checkpoint below
    else:
        bb_src = load_backbone_weights(cfg, bb, cfg.backbone_ckpt)
    F_dim = bb.num_features

    if cfg.grad_ckpt:
        try:
            bb.set_grad_checkpointing(True)
            print("[model] gradient checkpointing ON", flush=True)
        except Exception as e:
            print(f"[model] grad_ckpt unsupported for {cfg.arch}: {e}", flush=True)

    if cfg.freeze_stages > 0:
        pref = ("stem",) + tuple(f"stages.{i}." for i in range(cfg.freeze_stages))
        n_frozen = 0
        for nm, p in bb.named_parameters():
            if nm.startswith(pref):
                p.requires_grad = False
                n_frozen += p.numel()
        print(f"[model] froze stem + first {cfg.freeze_stages} stage(s): {n_frozen / 1e6:.2f}M params", flush=True)

    head_cfg = dict(F_dim=F_dim, dim=cfg.tf_dim, depth=cfg.tf_depth, heads=cfg.tf_heads,
                    n_slots=N_SLOTS, drop=cfg.head_drop, tf_drop=cfg.tf_drop)
    head = KneeHeadTF(**head_cfg)
    if cfg.resume_ckpt:
        pass                                                         # head is loaded together with the backbone below
    elif cfg.head_ckpt:
        hk = torch.load(cfg.head_ckpt, map_location="cpu", weights_only=False)
        if hk.get("head_cfg") is not None:
            for key in ("F_dim", "dim", "depth", "heads", "n_slots"):
                assert hk["head_cfg"][key] == head_cfg[key], \
                    f"head_ckpt {key}={hk['head_cfg'][key]} != config {head_cfg[key]}"
        print("[head] init from phase-1 checkpoint:", head.load_state_dict(hk["head"], strict=True), flush=True)
    else:
        print("[head] random init (no head_ckpt) -- consider head_lr ~5e-4", flush=True)

    model = KneeModelE2E(bb, head, amp=cfg.amp and dev == "cuda")
    if cfg.resume_ckpt:
        assert os.path.exists(cfg.resume_ckpt), f"resume_ckpt not found: {cfg.resume_ckpt}"
        ck = torch.load(cfg.resume_ckpt, map_location="cpu", weights_only=False)
        sd = ck["model"] if isinstance(ck, dict) and "model" in ck else ck
        sd = {(k[len("module."):] if k.startswith("module.") else k): v for k, v in sd.items()}
        if isinstance(ck, dict):
            if ck.get("arch") and ck["arch"] != cfg.arch:
                raise ValueError(f"resume checkpoint arch '{ck['arch']}' != cfg.arch '{cfg.arch}'")
            if ck.get("res") and ck["res"] != cfg.res:
                print(f"[resume] WARNING: checkpoint res {ck['res']} != cfg.res {cfg.res}", flush=True)
            if ck.get("head_cfg") is not None:
                for key in ("F_dim", "dim", "depth", "heads", "n_slots"):
                    assert ck["head_cfg"][key] == head_cfg[key], \
                        f"resume ckpt head {key}={ck['head_cfg'][key]} != config {head_cfg[key]}"
        print("[resume] load_state_dict:", model.load_state_dict(sd, strict=True), flush=True)
        ep_src = ck.get("epoch", "?") if isinstance(ck, dict) else "?"
        auc_src = ck.get("gold_auc") if isinstance(ck, dict) else None
        print(f"[resume] loaded {cfg.resume_ckpt} | source epoch {ep_src}"
              + (f" | its recorded gold macro-AUC {auc_src:.4f}" if auc_src is not None else ""), flush=True)
    model = model.to(dev)
    n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"[model] {cfg.arch} + KneeHeadTF | trainable {n_tr / 1e6:.2f}M | src {bb_src}", flush=True)
    return model, head_cfg, bb_src


# ==============================================================================
# 7. TRAIN / EVALUATE
# ==============================================================================
def macro_auc(P, Y):
    aucs = {name: float(roc_auc_score(Y[:, j], P[:, j]))
            for j, name in enumerate(LAB) if len(set(Y[:, j].astype(int))) > 1}
    return float(np.mean(list(aucs.values()))), aucs


@torch.no_grad()
def predict(model, loader, dev, desc="eval"):
    """loader must yield ONE study per batch with all its windows (eval dataset, batch_size=1)."""
    model.eval()
    P = []
    for x, slot, pos, pad, _ in tqdm(loader, desc=desc, leave=False, mininterval=5):
        o = torch.sigmoid(model(x.to(dev).float(), slot.to(dev), pos.to(dev), pad.to(dev)).float())
        P.append(o.cpu().numpy())
    return np.concatenate(P)


def evaluate(model, loader, Y, dev, desc="eval"):
    P = predict(model, loader, dev, desc=desc)
    au, aucs = macro_auc(P, Y)
    return au, aucs, P


def model_state_cpu(model):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def ckpt_meta(cfg, head_cfg, bb_src):
    return dict(src=bb_src, res=cfg.res, arch=cfg.arch, norm=cfg.norm, lab=LAB, head_cfg=head_cfg,
                slots=SLOTS, win=WIN, win_stride=WIN_STRIDE, aug=cfg.aug, resume_from=cfg.resume_ckpt)


def save_best(cfg, run, head_cfg, bb_src, ep, partial):
    best = run.best_state
    torch.save({**ckpt_meta(cfg, head_cfg, bb_src), "model": best["model"], "gold_auc": run.best,
                "aucs": best["aucs"], "epoch": best["epoch"]},
               os.path.join(cfg.HERE, f"knee_{cfg.tag}.tmp"))
    os.replace(os.path.join(cfg.HERE, f"knee_{cfg.tag}.tmp"), os.path.join(cfg.HERE, f"knee_{cfg.tag}.pt"))
    np.savez(os.path.join(cfg.HERE, f"knee_gold_{cfg.tag}.npz"),
             pred=run.best_P, truth=run.val_y, ids=np.array(run.gold_ids))
    summary = {"tag": cfg.tag, "src": bb_src, "best_gold_auc": run.best, "aucs": best["aucs"], "hist": run.hist,
               "res": cfg.res, "epochs": cfg.epochs, "bb_lr": cfg.bb_lr, "head_lr": cfg.head_lr,
               "bs": cfg.bs, "accum_steps": cfg.accum_steps, "n_gpu": cfg.n_gpu, "bs_per_gpu": cfg.bs_per_gpu,
               "k_train": cfg.k_train, "head_cfg": head_cfg,
               "aug": cfg.aug, "aug_params": dict(p=cfg.aug_p, rot=cfg.aug_rot, scale=cfg.aug_scale,
                                                  shift=cfg.aug_shift, gamma=cfg.aug_gamma, gain=cfg.aug_gain),
               "drop_path": cfg.drop_path, "baseline_gold_auc": run.baseline,
               "n_train": run.n_train, "n_gold": len(run.gold_ids), "arch": cfg.arch,
               "resume_ckpt": cfg.resume_ckpt, "partial": partial, "epochs_done": ep + 1}
    with open(os.path.join(cfg.HERE, f"knee_{cfg.tag}.json"), "w") as f:
        json.dump(summary, f, indent=1)


def train(cfg, model, head_cfg, bb_src, data, id2row, dev):
    mk = lambda ids, k, train: StudyWindowsE2E(cfg.corpus_root, ids, id2row, data.labels,
                                               cfg.res, k, train, cfg.norm)
    tds = mk(data.train_ids, cfg.k_train, True)
    vds = mk(data.gold_ids, 0, False)
    tl = DataLoader(tds, batch_size=cfg.bs, shuffle=True, num_workers=cfg.workers, drop_last=True,
                    pin_memory=True, persistent_workers=cfg.workers > 0)
    vl = DataLoader(vds, batch_size=1, shuffle=False, num_workers=min(cfg.workers, 2))
    val_y = np.stack([data.labels[u] for u in data.gold_ids]).astype(np.float32)

    bb_params = [p for p in model.backbone.parameters() if p.requires_grad]
    opt = torch.optim.AdamW([{"params": bb_params, "lr": cfg.bb_lr},
                             {"params": model.head.parameters(), "lr": cfg.head_lr}], weight_decay=cfg.wd)
    opt_steps = max(len(tl) // cfg.accum_steps, 1)
    n_batches = opt_steps * cfg.accum_steps
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[cfg.bb_lr, cfg.head_lr],
                                                total_steps=opt_steps * cfg.epochs, pct_start=cfg.pct_start)
    scaler = torch.amp.GradScaler("cuda", enabled=cfg.amp and dev == "cuda")
    lossf = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(data.pos_weight, device=dev))

    # `model` stays the plain module (state_dict keys, eval, checkpoints, SWA). Only training goes through the
    # DataParallel wrapper (when multi_gpu=True), which shares the same parameters and splits each micro-batch.
    train_model = nn.DataParallel(model, device_ids=list(range(cfg.n_gpu))) if cfg.n_gpu > 1 else model

    run = SimpleNamespace(best=0.0, best_state=None, best_P=None, hist=[], topk=[], val_y=val_y,
                          gold_ids=data.gold_ids, n_train=len(data.train_ids), baseline=None, val_loader=vl)
    print(f"[train] aug {'ON' if cfg.aug else 'OFF'} | {cfg.n_gpu} GPU(s) | micro-batch {cfg.bs} x accum "
          f"{cfg.accum_steps} | k_train {cfg.k_train} | {n_batches} micro-batches / {opt_steps} optimizer "
          f"steps per epoch | tag {cfg.tag}", flush=True)

    if cfg.eval_baseline:
        t = time.time()
        run.baseline, _, _ = evaluate(model, vl, val_y, dev, desc="baseline eval")
        print(f"[baseline] {'RESUMED' if cfg.resume_ckpt else 'warm-started'} model gold macro-AUC "
              f"{run.baseline:.4f} ({time.time() - t:.0f}s)  <- epochs below are compared against this", flush=True)

    TOPK = max(1, cfg.topk)
    t0 = time.time()
    for ep in range(cfg.epochs):
        train_model.train()
        tot, n_loss = 0.0, 0
        opt.zero_grad(set_to_none=True)
        te = time.time()
        pbar = tqdm(tl, total=n_batches, desc=f"ep{ep}", mininterval=5)
        for it, (x, slot, pos, pad, y) in enumerate(pbar):
            if it >= n_batches:
                break
            x = x.to(dev, non_blocking=True).float()
            slot, pos, pad, y = (t.to(dev, non_blocking=True) for t in (slot, pos, pad, y))
            if cfg.aug:
                x = series_shared_augment(x, slot, cfg)
            loss = lossf(train_model(x, slot, pos, pad).float(), y) / cfg.accum_steps
            scaler.scale(loss).backward()
            tot += loss.item() * cfg.accum_steps
            n_loss += 1
            if it % 10 == 0:
                pbar.set_postfix(loss=f"{tot / max(n_loss, 1):.3f}")
            if it % PROGRESS_EVERY == 0:  # plain-text fallback in case the bar doesn't render in the saved log
                spi = (time.time() - te) / (it + 1)
                print(f"  ep{ep} it {it}/{n_batches} | loss {tot / max(n_loss, 1):.3f} | {spi:.2f}s/it "
                      f"| est. epoch {spi * n_batches / 60:.1f} min", flush=True)
            if (it + 1) % cfg.accum_steps == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.clip_grad)
                scaler.step(opt)
                scaler.update()
                opt.zero_grad(set_to_none=True)
                sched.step()
        pbar.close()
        t_train = time.time() - te

        au, aucs, P = evaluate(model, vl, val_y, dev, desc=f"ep{ep} gold eval")
        run.hist.append({"ep": ep, "loss": tot / max(n_loss, 1), "gold_auc": au})

        if au > run.best:
            run.best, run.best_P = au, P
            run.best_state = {"model": model_state_cpu(model), "aucs": aucs, "epoch": ep}
            save_best(cfg, run, head_cfg, bb_src, ep, partial=True)

        if len(run.topk) < TOPK or au > min(t["gold_auc"] for t in run.topk):
            run.topk.append({"model": model_state_cpu(model), "gold_auc": au, "aucs": aucs, "epoch": ep, "P": P})
            run.topk.sort(key=lambda t: -t["gold_auc"])
            del run.topk[TOPK:]

        star = "  * NEW BEST" if (au >= run.best and run.best_state["epoch"] == ep) else ""
        d_base = f" | vs baseline {au - run.baseline:+.4f}" if run.baseline is not None else ""
        print("=" * 92, flush=True)
        print(f"EPOCH {ep + 1}/{cfg.epochs} | train loss {tot / max(n_loss, 1):.4f} | "
              f"GOLD macro-AUC {au:.4f} | best {run.best:.4f} (ep{run.best_state['epoch']}){d_base}{star}", flush=True)
        print("  per-finding: " + " | ".join(f"{k} {v:.3f}" for k, v in aucs.items()), flush=True)
        print(f"  time: train {t_train:.0f}s | total {time.time() - t0:.0f}s", flush=True)
        print("=" * 92, flush=True)

    print(f"\nDONE {cfg.tag} | BEST GOLD macro-AUC {run.best:.4f}"
          + (f" (baseline {run.baseline:.4f})" if run.baseline is not None else ""), flush=True)
    for k, v in run.best_state["aucs"].items():
        print(f"   {k:18s} {v:.3f}", flush=True)
    save_best(cfg, run, head_cfg, bb_src, cfg.epochs - 1, partial=False)
    print(f"saved knee_{cfg.tag}.pt / knee_gold_{cfg.tag}.npz / knee_{cfg.tag}.json", flush=True)
    return run


# ==============================================================================
# 8. POST-TRAINING: OOF, SWA (weight average of top-K epochs), EPOCH ENSEMBLE
# ==============================================================================
def post_training(cfg, model, head_cfg, bb_src, data, id2row, run, dev):
    vl, val_y = run.val_loader, run.val_y
    model.load_state_dict(run.best_state["model"])

    if cfg.folds > 0 and data.oof_ids:
        ods = StudyWindowsE2E(cfg.corpus_root, data.oof_ids, id2row, data.labels, cfg.res, 0, False, cfg.norm)
        ol = DataLoader(ods, batch_size=1, shuffle=False, num_workers=min(cfg.workers, 2))
        Po = predict(model, ol, dev, desc="OOF eval")
        Yo = np.stack([data.labels[u] for u in data.oof_ids]).astype(np.float32)
        np.savez(os.path.join(cfg.HERE, f"knee_oof_{cfg.tag}_fold{cfg.fold}.npz"),
                 pred=Po, truth=Yo, ids=np.array(data.oof_ids), fold=cfg.fold, nfolds=cfg.folds)
        print(f"[cv] wrote OOF for best epoch ({len(data.oof_ids)} studies)", flush=True)

    if len(run.topk) < 2:
        return

    sds = [t["model"] for t in run.topk]
    avg = {}
    for k in sds[0]:
        v0 = sds[0][k]
        avg[k] = (sum(sd[k].double() for sd in sds) / len(sds)).to(v0.dtype) if v0.is_floating_point() \
            else v0.clone()
    model.load_state_dict(avg)
    au_swa, aucs_swa, P_swa = evaluate(model, vl, val_y, dev, desc="SWA eval")
    eps = [int(t["epoch"]) for t in run.topk]
    torch.save({**ckpt_meta(cfg, head_cfg, bb_src), "model": avg, "gold_auc": au_swa, "aucs": aucs_swa,
                "epoch": eps, "swa_over": eps}, os.path.join(cfg.HERE, f"knee_{cfg.tag}_swa.pt"))
    np.savez(os.path.join(cfg.HERE, f"knee_gold_{cfg.tag}_swa.npz"),
             pred=P_swa, truth=val_y, ids=np.array(run.gold_ids))
    print(f"SWA over epochs {eps} | gold {au_swa:.4f} (best-epoch {run.best:.4f}) "
          f"{'BETTER' if au_swa > run.best else 'no gain'}", flush=True)

    Pens = np.mean([t["P"] for t in run.topk], axis=0)
    ens, _ = macro_auc(Pens, val_y)
    np.savez(os.path.join(cfg.HERE, f"knee_gold_{cfg.tag}_epens.npz"),
             pred=Pens, truth=val_y, ids=np.array(run.gold_ids))
    print(f"TOPK epochs {eps} | best {run.best:.4f} | epoch-ensemble {ens:.4f} "
          f"({'BETTER' if ens > run.best else 'no gain'})", flush=True)
    model.load_state_dict(run.best_state["model"])


# ==============================================================================
# 9. MAIN
# ==============================================================================
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def main():
    cfg = apply_cli_overrides(Config())
    seed_everything(cfg.seed)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    setup_gpus(cfg)
    # Eval feeds a different number of windows per study; with benchmark=True cuDNN would autotune the whole
    # network again for every new shape, which can stall the baseline / epoch-end evaluation for a very long time.
    torch.backends.cudnn.benchmark = False
    print(f"device {dev} | arch {cfg.arch} | res {cfg.res} | norm {cfg.norm} | "
          f"aug {'ON' if cfg.aug else 'OFF'} | tag {cfg.tag}", flush=True)

    labels_csv = cfg.labels_csv or prepare_labels(cfg)
    ids = open_ids(cfg.corpus_root)
    id2row = {uid: i for i, uid in enumerate(ids)}
    print(f"[data] corpus ids: {len(ids)}", flush=True)
    data = build_label_sets(cfg, ids, labels_csv)

    model, head_cfg, bb_src = build_model(cfg, dev)
    run = train(cfg, model, head_cfg, bb_src, data, id2row, dev)
    post_training(cfg, model, head_cfg, bb_src, data, id2row, run, dev)


if __name__ == "__main__":
    main()

[windows] 70 windows | per slot: sag[0:22]=20, sag[22:40]=16, cor[40:55]=13, cor[55:65]=8, ax[65:80]=13
[gpu] using 1 of 2 GPU(s): Tesla T4 | micro-batch 4 studies (4/GPU) x accum 2 = 8 studies per step
device cuda | arch convnext_tiny.in12k_ft_in1k | res 224 | norm imagenet | aug ON | tag e2e_aug
[data] corpus ids: 4407
[data] gold-labeled studies present in corpus: 58
[data] train 4349 | gold-val 58
[ckpt] backbone <- knee_ft_dev.pt@ep8 | 180 tensors, unexpected 0
[model] gradient checkpointing ON
[head] random init (no head_ckpt) -- consider head_lr ~5e-4
[model] convnext_tiny.in12k_ft_in1k + KneeHeadTF | trainable 29.21M | src knee_ft_dev.pt@ep8
[train] aug ON | 1 GPU(s) | micro-batch 4 x accum 2 | k_train 24 | 1086 micro-batches / 543 optimizer steps per epoch | tag e2e_aug


baseline eval:   0%|          | 0/58 [00:00<?, ?it/s]

[baseline] warm-started model gold macro-AUC 0.5348 (25s)  <- epochs below are compared against this


ep0:   0%|          | 0/1086 [00:00<?, ?it/s]

  ep0 it 0/1086 | loss 1.053 | 20.16s/it | est. epoch 364.9 min
  ep0 it 50/1086 | loss 1.080 | 1.12s/it | est. epoch 20.3 min
  ep0 it 100/1086 | loss 1.071 | 0.85s/it | est. epoch 15.4 min
  ep0 it 150/1086 | loss 1.039 | 0.76s/it | est. epoch 13.8 min
  ep0 it 200/1086 | loss 1.024 | 0.71s/it | est. epoch 12.9 min
  ep0 it 250/1086 | loss 1.013 | 0.68s/it | est. epoch 12.4 min
  ep0 it 300/1086 | loss 1.001 | 0.67s/it | est. epoch 12.1 min
  ep0 it 350/1086 | loss 0.992 | 0.65s/it | est. epoch 11.8 min
  ep0 it 400/1086 | loss 0.985 | 0.64s/it | est. epoch 11.6 min
  ep0 it 450/1086 | loss 0.982 | 0.64s/it | est. epoch 11.5 min
  ep0 it 500/1086 | loss 0.980 | 0.63s/it | est. epoch 11.4 min
  ep0 it 550/1086 | loss 0.978 | 0.62s/it | est. epoch 11.3 min
  ep0 it 600/1086 | loss 0.976 | 0.62s/it | est. epoch 11.2 min
  ep0 it 650/1086 | loss 0.973 | 0.62s/it | est. epoch 11.1 min
  ep0 it 700/1086 | loss 0.974 | 0.61s/it | est. epoch 11.1 min
  ep0 it 750/1086 | loss 0.974 | 0.61s/it

ep0 gold eval:   0%|          | 0/58 [00:00<?, ?it/s]

... (43 outputs left out)

ep11 gold eval:   0%|          | 0/58 [00:00<?, ?it/s]

EPOCH 12/12 | train loss 0.8881 | GOLD macro-AUC 0.9175 | best 0.9189 (ep7) | vs baseline +0.3826
  per-finding: ACL 0.952 | MCL 0.973 | Medial Meniscus 0.971 | Lateral Meniscus 0.812 | Medial OA 0.984 | Lateral OA 0.839 | PF OA 0.833 | Effusion 0.990 | Synovitis 0.829 | Baker's 0.987 | Contusion 0.891 | Fracture 0.947
  time: train 631s | total 7692s

DONE e2e_aug | BEST GOLD macro-AUC 0.9189 (baseline 0.5348)
   ACL                0.956
   MCL                0.961
   Medial Meniscus    0.974
   Lateral Meniscus   0.853
   Medial OA          0.986
   Lateral OA         0.841
   PF OA              0.843
   Effusion           0.988
   Synovitis          0.828
   Baker's            0.989
   Contusion          0.880
   Fracture           0.928
saved knee_e2e_aug.pt / knee_gold_e2e_aug.npz / knee_e2e_aug.json


SWA eval:   0%|          | 0/58 [00:00<?, ?it/s]

SWA over epochs [7, 11, 10] | gold 0.9190 (best-epoch 0.9189) BETTER
TOPK epochs [7, 11, 10] | best 0.9189 | epoch-ensemble 0.9194 (BETTER)


In [3]:
# #!/usr/bin/env python3
# """RSNA Knee Abnormality Detection -- PHASE 2: end-to-end training (backbone + transformer head)
# with OPTIONAL series-shared augmentation.

# Phase 1 trained only the transformer head on cached features of a frozen backbone. Phase 2 unfreezes
# the backbone and trains everything together, warm-starting from the phase-1 weights:

#     backbone  <- Config.backbone_ckpt   (the same fine-tuned backbone phase 1 used)
#     head      <- Config.head_ckpt       (knee_head_*.pt written by phase 1; optional but recommended)

# Series-shared augmentation (Config.aug = True/False, or --aug 1 / --aug 0 on the command line)
#     Every window of one SERIES (= one slot of one study) receives the SAME random draw of rotation,
#     scale, shift, gamma and gain, so the geometry stays consistent across the slices of a series
#     (the 3 channels of each 2.5D window and all windows of the slot move together). Different slots
#     of the same study get independent draws. Augmentation runs on the GPU, only while training.
#     With aug off, training sees the unmodified windows (random window subset per study only), which
#     makes a clean A/B comparison. The run tag is chosen automatically (e2e_aug / e2e_noaug) so the two
#     runs never overwrite each other.

# Changes in this version
#     * single GPU by default (multi_gpu = False)
#     * tqdm progress bars for training and for every evaluation, plus a plain-text s/it print
#       every PROGRESS_EVERY iterations (reliable in "Save & Run All" logs)
#     * torch.backends.cudnn.benchmark = False (eval uses a different number of windows per study,
#       so autotune would re-benchmark the whole network on every new shape)

# Sections
#   0. Config / CLI                5. Series-shared augmentation (GPU)
#   1. Label preparation           6. Model (backbone + KneeHeadTF)
#   2. Slot layout + window meta   7. Train / evaluate
#   3. Corpus loading              8. Post-training (OOF, SWA, epoch ensemble)
#   4. Dataset + label sets        9. main()
# """
# import os
# import re
# import glob
# import json
# import math
# import time
# import random
# import argparse
# from types import SimpleNamespace

# import numpy as np
# import pandas as pd
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# from torch.utils.data import Dataset, DataLoader
# from sklearn.metrics import roc_auc_score
# from tqdm.auto import tqdm
# import timm


# # ==============================================================================
# # 0. CONFIG / CLI
# # ==============================================================================
# LAB = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA", "PF OA",
#        "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]

# PROGRESS_EVERY = 50             # plain-text progress line every N micro-batches


# class Config:
#     # ---- paths -------------------------------------------------------------
#     HERE = "/kaggle/working"
#     comp_train_csv = "/kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv"
#     corpus_root = "/kaggle/input/datasets/chikonzeroselemani/rsna-corpus/corpus_experiment"
#     backbone_ckpt = ("/kaggle/input/datasets/chikonzeroselemani/coatnet-fature-extractor/"
#                      "convnext_tiny_knee_backbone.pt")
#     head_ckpt = None            # path to knee_head_<tag>.pt from phase 1 (recommended). None -> random head
#     labels_csv = None           # path to an existing new_labels.csv; None -> rebuild it from the sources below

#     # ---- label sources (only used when labels_csv is None) ------------------
#     oof_pseudolabels = "/kaggle/input/datasets/chikonzeroselemani/pseudolabelsnew/pseudoLabelsNew.txt"
#     pseudolabel_files = [
#         "/kaggle/input/datasets/chikonzeroselemani/rsna-pseudolabels/train_pseudolabels.csv",
#         "/kaggle/input/datasets/chikonzeroselemani/rsnapseudolabel2/RSNApseudolabels2.txt",
#         "/kaggle/input/datasets/chikonzeroselemani/rsnapseudolabels3/RSNApseudolabels3.txt",
#         "/kaggle/input/datasets/chikonzeroselemani/rsnapseudolabels4/RSNApseudolabels4.txt",
#     ]
#     llm_weight = 0.5

#     # ---- model -------------------------------------------------------------
#     arch = "convnext_tiny.in12k_ft_in1k"
#     res = 224
#     norm = "none"               # "none" | "imagenet"  (must match how the backbone was trained)
#     drop_path = 0.0             # stochastic depth for the backbone (0 = off; 0.1 is a common fine-tune value)
#     freeze_stages = 0           # freeze stem + first N backbone stages (saves memory/time)
#     grad_ckpt = True            # gradient checkpointing in the backbone (slower, much less memory)
#     tf_dim = 256
#     tf_depth = 2
#     tf_heads = 8
#     tf_drop = 0.1
#     head_drop = 0.2

#     # ---- training ----------------------------------------------------------
#     epochs = 12
#     multi_gpu = False           # GPU 0 only (True -> nn.DataParallel over every visible GPU)
#     bs_per_gpu = 4              # studies per GPU per micro-batch (memory is set by this, not by the GPU count)
#     target_batch = 8            # studies per optimizer step; gradient accumulation is derived from it
#     bs = None                   # (derived) total studies per micro-batch  = bs_per_gpu * n_gpu
#     accum_steps = None          # (derived) = max(1, round(target_batch / bs))
#     n_gpu = 1                   # (derived) number of GPUs actually used
#     k_train = 24                # random windows per study at train time (eval uses ALL windows)
#     bb_lr = 2e-4
#     head_lr = 2e-4              # lowered because the head is warm-started; use ~5e-4 for a fresh head
#     wd = 0.02
#     pct_start = 0.1
#     clip_grad = 3.0
#     topk = 3
#     workers = 4
#     amp = True                  # fp16 autocast for the backbone (T4 has no bf16 conv engine)

#     # ---- series-shared augmentation ----------------------------------------
#     aug = True                  # <-- main switch (or --aug 0/1)
#     aug_p = 1.0                 # probability that a given series is augmented at all
#     aug_rot = 12.0              # max rotation, degrees  (uniform in +-aug_rot)
#     aug_scale = 0.10            # scale in 1 +- aug_scale
#     aug_shift = 0.06            # translation in +-aug_shift (fraction of half-width)
#     aug_gamma = 0.25            # gamma = exp(+-aug_gamma)
#     aug_gain = 0.10             # gain in 1 +- aug_gain

#     # ---- misc --------------------------------------------------------------
#     tag = None                  # None -> "e2e_aug" / "e2e_noaug"
#     seed = 42
#     smoke = False
#     limit = 0                   # >0 -> only the first `limit` training studies
#     eval_baseline = True        # evaluate the warm-started model once before training (sanity check)

#     # ---- cross-validation (optional) ---------------------------------------
#     folds = 0
#     fold = -1
#     fold_file = None            # json: {"folds": {study_uid: fold_index}}


# def apply_cli_overrides(cfg):
#     """Tiny CLI so the aug on/off comparison needs no code edit. Unknown args (Jupyter) are ignored."""
#     p = argparse.ArgumentParser(add_help=False)
#     p.add_argument("--aug", type=int, choices=[0, 1])
#     p.add_argument("--tag", type=str)
#     p.add_argument("--epochs", type=int)
#     p.add_argument("--smoke", action="store_true")
#     a, _ = p.parse_known_args()
#     if a.aug is not None:
#         cfg.aug = bool(a.aug)
#     if a.tag:
#         cfg.tag = a.tag
#     if a.epochs:
#         cfg.epochs = a.epochs
#     if a.smoke:
#         cfg.smoke = True
#     if cfg.smoke:
#         cfg.epochs, cfg.limit, cfg.workers, cfg.k_train = 2, 24, 0, 8
#     if cfg.tag is None:
#         cfg.tag = "e2e_aug" if cfg.aug else "e2e_noaug"
#     return cfg


# def setup_gpus(cfg):
#     """Decide how many GPUs to use and derive the batch size / gradient accumulation from it.

#     With nn.DataParallel the micro-batch of `bs` studies is split evenly across the GPUs, so each GPU
#     sees `bs_per_gpu` studies: per-GPU memory stays the same while throughput roughly doubles on 2 GPUs.
#     The optimizer-step batch stays ~target_batch studies, so learning rates keep their meaning."""
#     avail = torch.cuda.device_count()
#     cfg.n_gpu = max(1, avail) if cfg.multi_gpu else 1
#     cfg.bs = cfg.bs_per_gpu * cfg.n_gpu
#     cfg.accum_steps = max(1, round(cfg.target_batch / cfg.bs))
#     names = ", ".join(torch.cuda.get_device_name(i) for i in range(min(cfg.n_gpu, avail))) or "cpu"
#     print(f"[gpu] using {cfg.n_gpu} of {avail} GPU(s): {names} | micro-batch {cfg.bs} studies "
#           f"({cfg.bs_per_gpu}/GPU) x accum {cfg.accum_steps} = {cfg.bs * cfg.accum_steps} studies per step",
#           flush=True)
#     return cfg


# # ==============================================================================
# # 1. LABEL PREPARATION
# # ==============================================================================
# def prepare_labels(cfg):
#     """Blend LLM soft labels with pseudo-labels, append gold rows, write new_labels.csv."""
#     soft = pd.read_csv(cfg.oof_pseudolabels)
#     soft["StudyInstanceUID"] = soft["StudyInstanceUID"].astype(str)

#     pseudo = pd.concat([pd.read_csv(p) for p in cfg.pseudolabel_files], ignore_index=True)
#     pseudo["StudyInstanceUID"] = pseudo["StudyInstanceUID"].astype(str)
#     pseudo = pseudo.drop_duplicates("StudyInstanceUID", keep="last").set_index("StudyInstanceUID")

#     w_llm, w_ps = cfg.llm_weight, 1.0 - cfg.llm_weight
#     print(f"[labels] LLM_WEIGHT {w_llm} | PSEUDO_WEIGHT {w_ps}", flush=True)
#     common = soft["StudyInstanceUID"].isin(pseudo.index)
#     for col in LAB:
#         llm_v = soft.loc[common, col].astype(float)
#         ps_v = soft.loc[common, "StudyInstanceUID"].map(pseudo[col]).astype(float)
#         soft.loc[common, col] = w_llm * llm_v + w_ps * ps_v
#     print(f"[labels] blended {int(common.sum())} / {len(soft)} studies", flush=True)

#     tr = pd.read_csv(cfg.comp_train_csv)
#     tr["StudyInstanceUID"] = tr["StudyInstanceUID"].astype(str)
#     gold_df = tr[tr[LAB].notna().all(axis=1)].copy()

#     cols = ["StudyInstanceUID"] + LAB
#     soft = pd.concat([soft[cols], gold_df[cols]], ignore_index=True).drop_duplicates("StudyInstanceUID", keep="last")
#     out = os.path.join(cfg.HERE, "new_labels.csv")
#     soft.to_csv(out, index=False)
#     print(f"[labels] wrote {out} ({len(soft)} rows)", flush=True)
#     return out


# # ==============================================================================
# # 2. SLOT LAYOUT + WINDOW METADATA  (identical to phase 1)
# # ==============================================================================
# SLOTS = [("sag", 0, 22), ("sag", 22, 40), ("cor", 40, 55), ("cor", 55, 65), ("ax", 65, 80)]
# N_SLICES = 80
# N_SLOTS = len(SLOTS)
# WIN = 3
# WIN_STRIDE = 1


# def build_windows(slots=SLOTS, win=WIN, stride=WIN_STRIDE):
#     """(win_idx [n_win, win], win_slot [n_win], win_pos [n_win] = position in own slot, 0..1)."""
#     idx, slot_of, pos = [], [], []
#     for s, (plane, lo, hi) in enumerate(slots):
#         assert hi - lo >= win, f"slot {s} ({plane} {lo}:{hi}) shorter than window size {win}"
#         starts = list(range(lo, hi - win + 1, stride))
#         span = max(len(starts) - 1, 1)
#         for i, start in enumerate(starts):
#             idx.append(list(range(start, start + win)))
#             slot_of.append(s)
#             pos.append(i / span)
#     return (np.asarray(idx, dtype=np.int64), np.asarray(slot_of, dtype=np.int64),
#             np.asarray(pos, dtype=np.float32))


# WIN_IDX, WIN_SLOT, WIN_POS = build_windows()
# MAX_C = len(WIN_IDX)
# for _w, _s in zip(WIN_IDX, WIN_SLOT):
#     assert SLOTS[_s][1] <= _w.min() and _w.max() < SLOTS[_s][2], "window leaks out of its slot"
# print(f"[windows] {MAX_C} windows | per slot: "
#       + ", ".join(f"{p}[{lo}:{hi}]={n}" for (p, lo, hi), n in zip(SLOTS, np.bincount(WIN_SLOT))), flush=True)


# # ==============================================================================
# # 3. CORPUS LOADING  (identical to phase 1)
# # ==============================================================================
# _NPY = re.compile(r"^(?P<prefix>.*?)_?(?P<kind>ids|vols|masks)\.npy$")


# def _discover_parts(root):
#     groups = {}
#     found = sorted(glob.glob(os.path.join(str(root), "**", "*.npy"), recursive=True))
#     for f in found:
#         m = _NPY.match(os.path.basename(f))
#         if m:
#             groups.setdefault((os.path.dirname(f), m["prefix"]), {})[m["kind"]] = f
#     parts = [g for _, g in sorted(groups.items()) if "ids" in g and "vols" in g]
#     if not parts:
#         raise FileNotFoundError(f"No <prefix>_ids.npy + <prefix>_vols.npy pair under {root}.")
#     return parts


# class MultiPartVols:
#     """Several memory-mapped (n_i, N_SLICES, H, W) arrays presented as one array."""

#     def __init__(self, parts):
#         self.parts = parts
#         self.offsets = np.cumsum([0] + [p.shape[0] for p in parts])
#         self.shape = (int(self.offsets[-1]),) + tuple(parts[0].shape[1:])

#     def __len__(self):
#         return self.shape[0]

#     def __getitem__(self, row):
#         j = int(np.searchsorted(self.offsets, row, side="right") - 1)
#         return self.parts[j][row - self.offsets[j]]


# def open_ids(root):
#     return np.concatenate([np.load(g["ids"], allow_pickle=True).astype(str) for g in _discover_parts(root)])


# def open_corpus(root):
#     parts = _discover_parts(root)
#     vols_parts = []
#     for g in parts:
#         v = np.load(g["vols"], mmap_mode="r")
#         n_ids = np.load(g["ids"], allow_pickle=True).shape[0]
#         assert v.ndim == 4 and v.shape[1] == N_SLICES, \
#             f"{os.path.basename(g['vols'])}: expected (N, {N_SLICES}, H, W), got {v.shape}"
#         assert v.shape[0] == n_ids, f"{os.path.basename(g['vols'])}: {v.shape[0]} volumes but {n_ids} ids"
#         vols_parts.append(v)
#     vols = MultiPartVols(vols_parts)
#     masks = None
#     if all("masks" in g for g in parts):
#         masks = np.concatenate([np.load(g["masks"]) for g in parts], axis=0)
#         if masks.shape != (vols.shape[0], N_SLICES):
#             print(f"[corpus] WARNING: masks {masks.shape} != {(vols.shape[0], N_SLICES)} - ignoring masks", flush=True)
#             masks = None
#     return vols, masks


# def candidate_windows(vol, mask_row):
#     """Ordered ids (rows of WIN_IDX) of every slot-safe window whose 3 slices are all non-empty."""
#     if mask_row is not None:
#         valid = mask_row > 0
#     else:
#         valid = np.asarray(vol).reshape(N_SLICES, -1).any(axis=1)
#     cs = np.flatnonzero(valid[WIN_IDX].all(axis=1))
#     if len(cs) == 0:
#         cs = np.arange(len(WIN_IDX))
#     return cs


# # ==============================================================================
# # 4. DATASET + LABEL SETS
# # ==============================================================================
# class StudyWindowsE2E(Dataset):
#     """Per-study bag of 2.5D windows (3 consecutive slices of ONE slot as RGB), full image resized to `res`.

#     train=True : k random DISTINCT windows (sorted); studies with fewer than k get masked padding.
#     train=False: ALL windows of the study (use batch_size=1).
#     Returns (x [K,3,res,res] fp16 in 0..1, slot [K] long, pos [K] float, pad [K] bool, y [12]).
#     No augmentation happens here -- it is applied on the GPU (section 5) so it can be shared per series."""

#     def __init__(self, corpus_root, ids, id2row, labels, res, k, train, norm="none"):
#         self.corpus_root, self.ids, self.id2row, self.labels = corpus_root, ids, id2row, labels
#         self.res, self.k, self.train, self.norm = res, k, train, norm
#         self._mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
#         self._std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
#         self.vols = None
#         self.masks = None

#     def __len__(self):
#         return len(self.ids)

#     def _ensure(self):
#         if self.vols is None:
#             self.vols, self.masks = open_corpus(self.corpus_root)

#     def _resize(self, tri):
#         t = torch.from_numpy(np.ascontiguousarray(tri))
#         if t.shape[-1] != self.res or t.shape[-2] != self.res:
#             t = F.interpolate(t[None], size=(self.res, self.res), mode="bilinear", align_corners=False)[0]
#         return t

#     def __getitem__(self, i):
#         self._ensure()
#         uid = self.ids[i]
#         row = self.id2row[uid]
#         vol = self.vols[row]
#         cs = candidate_windows(vol, None if self.masks is None else self.masks[row])

#         if self.train:
#             if len(cs) > self.k:
#                 cs = np.array(sorted(random.sample(list(cs), self.k)), dtype=np.int64)
#             K = self.k
#         else:
#             K = len(cs)
#         n = len(cs)

#         x = torch.zeros(K, 3, self.res, self.res, dtype=torch.float16)
#         for j, w in enumerate(cs):
#             tri = np.asarray(vol[WIN_IDX[w]]).astype(np.float32)
#             if vol.dtype == np.uint8:
#                 tri /= 255.0
#             t = self._resize(tri)
#             if self.norm == "imagenet":
#                 t = (t - self._mean) / self._std
#             x[j] = t
#         slot = torch.zeros(K, dtype=torch.long)
#         pos = torch.zeros(K, dtype=torch.float32)
#         slot[:n] = torch.from_numpy(WIN_SLOT[cs])
#         pos[:n] = torch.from_numpy(WIN_POS[cs])
#         pad = torch.arange(K) >= n
#         y = torch.from_numpy(self.labels[uid])
#         return x, slot, pos, pad, y


# def build_label_sets(cfg, ids, labels_csv):
#     idset = set(ids)
#     tr = pd.read_csv(cfg.comp_train_csv)
#     tr["StudyInstanceUID"] = tr["StudyInstanceUID"].astype(str)
#     gold_df = tr[tr[LAB].notna().all(axis=1)].copy().set_index("StudyInstanceUID")
#     gold_ids = [u for u in gold_df.index if u in idset]
#     print(f"[data] gold-labeled studies present in corpus: {len(gold_ids)}", flush=True)

#     soft = pd.read_csv(labels_csv)
#     soft["StudyInstanceUID"] = soft["StudyInstanceUID"].astype(str)
#     soft = soft.set_index("StudyInstanceUID")

#     goldset = set(gold_ids)
#     train_ids = [u for u in ids if u in soft.index and u not in goldset]

#     oof_ids = []
#     if cfg.folds > 0:
#         assert 0 <= cfg.fold < cfg.folds, f"fold must be in [0,{cfg.folds}) when folds>0"
#         assert cfg.fold_file and os.path.exists(cfg.fold_file), "fold_file required"
#         with open(cfg.fold_file, "r") as f:
#             fmap = json.load(f)["folds"]
#         held = {u for u in train_ids if fmap.get(u, -1) == cfg.fold}
#         oof_ids = [u for u in train_ids if u in held]
#         train_ids = [u for u in train_ids if u not in held]
#         print(f"[cv] fold {cfg.fold}/{cfg.folds}: train {len(train_ids)} | OOF {len(oof_ids)}", flush=True)

#     if cfg.limit:
#         train_ids = train_ids[:cfg.limit]

#     labels = {u: soft.loc[u, LAB].values.astype(np.float32) for u in train_ids}
#     for u in oof_ids:
#         labels[u] = soft.loc[u, LAB].values.astype(np.float32)
#     for u in gold_ids:
#         labels[u] = gold_df.loc[u, LAB].values.astype(np.float32)
#     print(f"[data] train {len(train_ids)} | gold-val {len(gold_ids)}"
#           + (f" | oof {len(oof_ids)}" if oof_ids else ""), flush=True)

#     prev = np.clip(np.stack([labels[u] for u in train_ids]).mean(0), 0.03, 0.7)
#     pos_weight = np.clip((1 - prev) / prev, 1, 10).astype(np.float32)
#     return SimpleNamespace(train_ids=train_ids, gold_ids=gold_ids, oof_ids=oof_ids,
#                            labels=labels, pos_weight=pos_weight)


# # ==============================================================================
# # 5. SERIES-SHARED AUGMENTATION (GPU)
# # ==============================================================================
# @torch.no_grad()
# def series_shared_augment(x, slot, cfg):
#     """x [B,K,3,H,W] float in 0..1 (on GPU) | slot [B,K] long -> augmented x, same shape.

#     One random draw of (rotation, scale, shift x/y, gamma, gain) per SERIES = per (study, slot).
#     Every window of that series -- and all 3 channels inside each window -- gets the identical
#     affine warp and intensity change, so the stacked slices stay geometrically consistent.
#     Slots of the same study draw independently. Padded windows are all zeros and stay zeros."""
#     B, K = slot.shape
#     dev = x.device

#     def u():                                           # uniform in [-1, 1], one value per (study, slot)
#         return torch.rand(B, N_SLOTS, device=dev) * 2 - 1

#     on = (torch.rand(B, N_SLOTS, device=dev) < cfg.aug_p).float()   # series left untouched when 0
#     angle = u() * math.radians(cfg.aug_rot) * on
#     scale = 1 + u() * cfg.aug_scale * on
#     tx = u() * cfg.aug_shift * on
#     ty = u() * cfg.aug_shift * on
#     gamma = torch.exp(u() * cfg.aug_gamma * on)
#     gain = 1 + u() * cfg.aug_gain * on

#     def per_window(p):                                 # [B, N_SLOTS] -> [B*K], each window reads its series' value
#         return p.gather(1, slot).reshape(-1)

#     angle, scale, tx, ty, gamma, gain = map(per_window, (angle, scale, tx, ty, gamma, gain))
#     cos, sin = torch.cos(angle) * scale, torch.sin(angle) * scale
#     theta = torch.stack([torch.stack([cos, -sin, tx], 1), torch.stack([sin, cos, ty], 1)], 1)   # [B*K,2,3]

#     xf = x.flatten(0, 1)                                                                          # [B*K,3,H,W]
#     grid = F.affine_grid(theta, tuple(xf.shape), align_corners=False)
#     xf = F.grid_sample(xf, grid, mode="bilinear", padding_mode="zeros", align_corners=False)
#     xf = xf.clamp(0, 1).pow(gamma.view(-1, 1, 1, 1)) * gain.view(-1, 1, 1, 1)
#     return xf.clamp(0, 1).view_as(x)


# # ==============================================================================
# # 6. MODEL
# # ==============================================================================
# def build_backbone(cfg, pretrained=False):
#     is_vit = any(k in cfg.arch for k in ("vit", "deit", "dinov2", "eva", "beit"))
#     kw = dict(pretrained=pretrained, num_classes=0, in_chans=3)
#     if is_vit:
#         kw.update(global_pool="token", dynamic_img_size=True)
#     else:
#         kw.update(global_pool="avg")
#     if cfg.drop_path > 0:
#         kw["drop_path_rate"] = cfg.drop_path
#     return timm.create_model(cfg.arch, **kw)


# def load_backbone_weights(cfg, bb, ckpt_path):
#     """Accepts a feature-extractor file ({'backbone': bare sd, ...}) or a classifier checkpoint ('backbone.*')."""
#     ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
#     meta = {}
#     if isinstance(ck, dict) and isinstance(ck.get("backbone"), dict) \
#             and not any(k.startswith("backbone.") for k in ck):
#         bbst = ck["backbone"]
#         meta = {k: ck.get(k) for k in ("arch", "res", "feature_dim", "source_epoch", "source_ckpt")}
#         if meta.get("arch") and meta["arch"] != cfg.arch:
#             print(f"[ckpt] WARNING: file arch '{meta['arch']}' != cfg.arch '{cfg.arch}'", flush=True)
#         if meta.get("res") and meta["res"] != cfg.res:
#             print(f"[ckpt] WARNING: file res {meta['res']} != cfg.res {cfg.res}", flush=True)
#     else:
#         st = ck.get("student", ck.get("model", ck)) if isinstance(ck, dict) else ck
#         bbst = {k[len("backbone."):]: v for k, v in st.items() if k.startswith("backbone.")}
#         if not bbst:
#             bbst = {k: v for k, v in st.items() if isinstance(v, torch.Tensor)}
#         meta["source_epoch"] = ck.get("epoch", "?") if isinstance(ck, dict) else "?"
#     assert bbst, f"no backbone tensors found in {ckpt_path}"
#     missing, unexpected = bb.load_state_dict(bbst, strict=False)
#     assert not missing, f"backbone tensors missing from checkpoint: {missing[:5]} ..."
#     src = f"{meta.get('source_ckpt') or os.path.basename(ckpt_path)}@ep{meta.get('source_epoch', '?')}"
#     print(f"[ckpt] backbone <- {src} | {len(bbst)} tensors, unexpected {len(unexpected)}", flush=True)
#     return src


# class KneeHeadTF(nn.Module):
#     """window features -> Dropout/Linear/LayerNorm -> + slot emb + slice-position emb
#        -> transformer over ALL windows of the study -> per-finding attention pooling -> 12 logits."""

#     def __init__(self, F_dim, n=12, dim=256, depth=2, heads=8, n_slots=5, drop=0.2, tf_drop=0.1):
#         super().__init__()
#         self.proj = nn.Sequential(nn.Dropout(drop), nn.Linear(F_dim, dim), nn.LayerNorm(dim))
#         self.slot_emb = nn.Embedding(n_slots, dim)
#         self.pos_emb = nn.Sequential(nn.Linear(1, dim), nn.GELU(), nn.Linear(dim, dim))
#         layer = nn.TransformerEncoderLayer(dim, heads, dim * 2, dropout=tf_drop, activation="gelu",
#                                            batch_first=True, norm_first=True)
#         self.tf = nn.TransformerEncoder(layer, depth, enable_nested_tensor=False) if depth else None
#         self.norm = nn.LayerNorm(dim)
#         self.att = nn.Sequential(nn.Linear(dim, 256), nn.Tanh(), nn.Dropout(drop), nn.Linear(256, n))
#         self.clsW = nn.Parameter(torch.zeros(n, dim))
#         self.clsb = nn.Parameter(torch.zeros(n))
#         nn.init.trunc_normal_(self.clsW, std=0.02)

#     def forward(self, feats, slot, pos, pad):
#         h = self.proj(feats) + self.slot_emb(slot) + self.pos_emb(pos.unsqueeze(-1))
#         if self.tf is not None:
#             h = self.tf(h, src_key_padding_mask=pad)
#         h = self.norm(h)
#         a = self.att(h).masked_fill(pad.unsqueeze(-1), -1e4).softmax(1)
#         pooled = torch.einsum("bkn,bkd->bnd", a, h)
#         return (pooled * self.clsW).sum(-1) + self.clsb


# class KneeModelE2E(nn.Module):
#     """backbone (fp16 autocast, trainable) + KneeHeadTF (always fp32). State-dict keys: backbone.* / head.*"""

#     def __init__(self, backbone, head, amp=True):
#         super().__init__()
#         self.backbone = backbone
#         self.head = head
#         self.amp = amp

#     def forward(self, x, slot, pos, pad):
#         """x [B,K,3,H,W] float | slot, pos, pad [B,K] -> logits [B,12]."""
#         B, K = x.shape[:2]
#         with torch.autocast("cuda", dtype=torch.float16, enabled=self.amp and x.is_cuda):
#             f = self.backbone(x.flatten(0, 1))
#         return self.head(f.float().view(B, K, -1), slot, pos, pad)


# def build_model(cfg, dev):
#     bb = build_backbone(cfg, pretrained=False)
#     bb_src = load_backbone_weights(cfg, bb, cfg.backbone_ckpt)
#     F_dim = bb.num_features

#     if cfg.grad_ckpt:
#         try:
#             bb.set_grad_checkpointing(True)
#             print("[model] gradient checkpointing ON", flush=True)
#         except Exception as e:
#             print(f"[model] grad_ckpt unsupported for {cfg.arch}: {e}", flush=True)

#     if cfg.freeze_stages > 0:
#         pref = ("stem",) + tuple(f"stages.{i}." for i in range(cfg.freeze_stages))
#         n_frozen = 0
#         for nm, p in bb.named_parameters():
#             if nm.startswith(pref):
#                 p.requires_grad = False
#                 n_frozen += p.numel()
#         print(f"[model] froze stem + first {cfg.freeze_stages} stage(s): {n_frozen / 1e6:.2f}M params", flush=True)

#     head_cfg = dict(F_dim=F_dim, dim=cfg.tf_dim, depth=cfg.tf_depth, heads=cfg.tf_heads,
#                     n_slots=N_SLOTS, drop=cfg.head_drop, tf_drop=cfg.tf_drop)
#     head = KneeHeadTF(**head_cfg)
#     if cfg.head_ckpt:
#         hk = torch.load(cfg.head_ckpt, map_location="cpu", weights_only=False)
#         if hk.get("head_cfg") is not None:
#             for key in ("F_dim", "dim", "depth", "heads", "n_slots"):
#                 assert hk["head_cfg"][key] == head_cfg[key], \
#                     f"head_ckpt {key}={hk['head_cfg'][key]} != config {head_cfg[key]}"
#         print("[head] init from phase-1 checkpoint:", head.load_state_dict(hk["head"], strict=True), flush=True)
#     else:
#         print("[head] random init (no head_ckpt) -- consider head_lr ~5e-4", flush=True)

#     model = KneeModelE2E(bb, head, amp=cfg.amp and dev == "cuda").to(dev)
#     n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
#     print(f"[model] {cfg.arch} + KneeHeadTF | trainable {n_tr / 1e6:.2f}M | src {bb_src}", flush=True)
#     return model, head_cfg, bb_src


# # ==============================================================================
# # 7. TRAIN / EVALUATE
# # ==============================================================================
# def macro_auc(P, Y):
#     aucs = {name: float(roc_auc_score(Y[:, j], P[:, j]))
#             for j, name in enumerate(LAB) if len(set(Y[:, j].astype(int))) > 1}
#     return float(np.mean(list(aucs.values()))), aucs


# @torch.no_grad()
# def predict(model, loader, dev, desc="eval"):
#     """loader must yield ONE study per batch with all its windows (eval dataset, batch_size=1)."""
#     model.eval()
#     P = []
#     for x, slot, pos, pad, _ in tqdm(loader, desc=desc, leave=False, mininterval=5):
#         o = torch.sigmoid(model(x.to(dev).float(), slot.to(dev), pos.to(dev), pad.to(dev)).float())
#         P.append(o.cpu().numpy())
#     return np.concatenate(P)


# def evaluate(model, loader, Y, dev, desc="eval"):
#     P = predict(model, loader, dev, desc=desc)
#     au, aucs = macro_auc(P, Y)
#     return au, aucs, P


# def model_state_cpu(model):
#     return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


# def ckpt_meta(cfg, head_cfg, bb_src):
#     return dict(src=bb_src, res=cfg.res, arch=cfg.arch, norm=cfg.norm, lab=LAB, head_cfg=head_cfg,
#                 slots=SLOTS, win=WIN, win_stride=WIN_STRIDE, aug=cfg.aug)


# def save_best(cfg, run, head_cfg, bb_src, ep, partial):
#     best = run.best_state
#     torch.save({**ckpt_meta(cfg, head_cfg, bb_src), "model": best["model"], "gold_auc": run.best,
#                 "aucs": best["aucs"], "epoch": best["epoch"]},
#                os.path.join(cfg.HERE, f"knee_{cfg.tag}.tmp"))
#     os.replace(os.path.join(cfg.HERE, f"knee_{cfg.tag}.tmp"), os.path.join(cfg.HERE, f"knee_{cfg.tag}.pt"))
#     np.savez(os.path.join(cfg.HERE, f"knee_gold_{cfg.tag}.npz"),
#              pred=run.best_P, truth=run.val_y, ids=np.array(run.gold_ids))
#     summary = {"tag": cfg.tag, "src": bb_src, "best_gold_auc": run.best, "aucs": best["aucs"], "hist": run.hist,
#                "res": cfg.res, "epochs": cfg.epochs, "bb_lr": cfg.bb_lr, "head_lr": cfg.head_lr,
#                "bs": cfg.bs, "accum_steps": cfg.accum_steps, "n_gpu": cfg.n_gpu, "bs_per_gpu": cfg.bs_per_gpu,
#                "k_train": cfg.k_train, "head_cfg": head_cfg,
#                "aug": cfg.aug, "aug_params": dict(p=cfg.aug_p, rot=cfg.aug_rot, scale=cfg.aug_scale,
#                                                   shift=cfg.aug_shift, gamma=cfg.aug_gamma, gain=cfg.aug_gain),
#                "drop_path": cfg.drop_path, "baseline_gold_auc": run.baseline,
#                "n_train": run.n_train, "n_gold": len(run.gold_ids), "arch": cfg.arch,
#                "partial": partial, "epochs_done": ep + 1}
#     with open(os.path.join(cfg.HERE, f"knee_{cfg.tag}.json"), "w") as f:
#         json.dump(summary, f, indent=1)


# def train(cfg, model, head_cfg, bb_src, data, id2row, dev):
#     mk = lambda ids, k, train: StudyWindowsE2E(cfg.corpus_root, ids, id2row, data.labels,
#                                                cfg.res, k, train, cfg.norm)
#     tds = mk(data.train_ids, cfg.k_train, True)
#     vds = mk(data.gold_ids, 0, False)
#     tl = DataLoader(tds, batch_size=cfg.bs, shuffle=True, num_workers=cfg.workers, drop_last=True,
#                     pin_memory=True, persistent_workers=cfg.workers > 0)
#     vl = DataLoader(vds, batch_size=1, shuffle=False, num_workers=min(cfg.workers, 2))
#     val_y = np.stack([data.labels[u] for u in data.gold_ids]).astype(np.float32)

#     bb_params = [p for p in model.backbone.parameters() if p.requires_grad]
#     opt = torch.optim.AdamW([{"params": bb_params, "lr": cfg.bb_lr},
#                              {"params": model.head.parameters(), "lr": cfg.head_lr}], weight_decay=cfg.wd)
#     opt_steps = max(len(tl) // cfg.accum_steps, 1)
#     n_batches = opt_steps * cfg.accum_steps
#     sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[cfg.bb_lr, cfg.head_lr],
#                                                 total_steps=opt_steps * cfg.epochs, pct_start=cfg.pct_start)
#     scaler = torch.amp.GradScaler("cuda", enabled=cfg.amp and dev == "cuda")
#     lossf = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(data.pos_weight, device=dev))

#     # `model` stays the plain module (state_dict keys, eval, checkpoints, SWA). Only training goes through the
#     # DataParallel wrapper (when multi_gpu=True), which shares the same parameters and splits each micro-batch.
#     train_model = nn.DataParallel(model, device_ids=list(range(cfg.n_gpu))) if cfg.n_gpu > 1 else model

#     run = SimpleNamespace(best=0.0, best_state=None, best_P=None, hist=[], topk=[], val_y=val_y,
#                           gold_ids=data.gold_ids, n_train=len(data.train_ids), baseline=None, val_loader=vl)
#     print(f"[train] aug {'ON' if cfg.aug else 'OFF'} | {cfg.n_gpu} GPU(s) | micro-batch {cfg.bs} x accum "
#           f"{cfg.accum_steps} | k_train {cfg.k_train} | {n_batches} micro-batches / {opt_steps} optimizer "
#           f"steps per epoch | tag {cfg.tag}", flush=True)

#     if cfg.eval_baseline:
#         t = time.time()
#         run.baseline, _, _ = evaluate(model, vl, val_y, dev, desc="baseline eval")
#         print(f"[baseline] warm-started model gold macro-AUC {run.baseline:.4f} ({time.time() - t:.0f}s)", flush=True)

#     TOPK = max(1, cfg.topk)
#     t0 = time.time()
#     for ep in range(cfg.epochs):
#         train_model.train()
#         tot, n_loss = 0.0, 0
#         opt.zero_grad(set_to_none=True)
#         te = time.time()
#         pbar = tqdm(tl, total=n_batches, desc=f"ep{ep}", mininterval=5)
#         for it, (x, slot, pos, pad, y) in enumerate(pbar):
#             if it >= n_batches:
#                 break
#             x = x.to(dev, non_blocking=True).float()
#             slot, pos, pad, y = (t.to(dev, non_blocking=True) for t in (slot, pos, pad, y))
#             if cfg.aug:
#                 x = series_shared_augment(x, slot, cfg)
#             loss = lossf(train_model(x, slot, pos, pad).float(), y) / cfg.accum_steps
#             scaler.scale(loss).backward()
#             tot += loss.item() * cfg.accum_steps
#             n_loss += 1
#             if it % 10 == 0:
#                 pbar.set_postfix(loss=f"{tot / max(n_loss, 1):.3f}")
#             if it % PROGRESS_EVERY == 0:  # plain-text fallback in case the bar doesn't render in the saved log
#                 spi = (time.time() - te) / (it + 1)
#                 print(f"  ep{ep} it {it}/{n_batches} | loss {tot / max(n_loss, 1):.3f} | {spi:.2f}s/it "
#                       f"| est. epoch {spi * n_batches / 60:.1f} min", flush=True)
#             if (it + 1) % cfg.accum_steps == 0:
#                 scaler.unscale_(opt)
#                 torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.clip_grad)
#                 scaler.step(opt)
#                 scaler.update()
#                 opt.zero_grad(set_to_none=True)
#                 sched.step()
#         pbar.close()
#         t_train = time.time() - te

#         au, aucs, P = evaluate(model, vl, val_y, dev, desc=f"ep{ep} gold eval")
#         run.hist.append({"ep": ep, "loss": tot / max(n_loss, 1), "gold_auc": au})

#         if au > run.best:
#             run.best, run.best_P = au, P
#             run.best_state = {"model": model_state_cpu(model), "aucs": aucs, "epoch": ep}
#             save_best(cfg, run, head_cfg, bb_src, ep, partial=True)

#         if len(run.topk) < TOPK or au > min(t["gold_auc"] for t in run.topk):
#             run.topk.append({"model": model_state_cpu(model), "gold_auc": au, "aucs": aucs, "epoch": ep, "P": P})
#             run.topk.sort(key=lambda t: -t["gold_auc"])
#             del run.topk[TOPK:]

#         print(f"ep{ep} loss {tot / max(n_loss, 1):.3f} | GOLD macro-AUC {au:.4f} (best {run.best:.4f}) "
#               f"| train {t_train:.0f}s | total {time.time() - t0:.0f}s", flush=True)

#     print(f"\nDONE {cfg.tag} | BEST GOLD macro-AUC {run.best:.4f}"
#           + (f" (baseline {run.baseline:.4f})" if run.baseline is not None else ""), flush=True)
#     for k, v in run.best_state["aucs"].items():
#         print(f"   {k:18s} {v:.3f}", flush=True)
#     save_best(cfg, run, head_cfg, bb_src, cfg.epochs - 1, partial=False)
#     print(f"saved knee_{cfg.tag}.pt / knee_gold_{cfg.tag}.npz / knee_{cfg.tag}.json", flush=True)
#     return run


# # ==============================================================================
# # 8. POST-TRAINING: OOF, SWA (weight average of top-K epochs), EPOCH ENSEMBLE
# # ==============================================================================
# def post_training(cfg, model, head_cfg, bb_src, data, id2row, run, dev):
#     vl, val_y = run.val_loader, run.val_y
#     model.load_state_dict(run.best_state["model"])

#     if cfg.folds > 0 and data.oof_ids:
#         ods = StudyWindowsE2E(cfg.corpus_root, data.oof_ids, id2row, data.labels, cfg.res, 0, False, cfg.norm)
#         ol = DataLoader(ods, batch_size=1, shuffle=False, num_workers=min(cfg.workers, 2))
#         Po = predict(model, ol, dev, desc="OOF eval")
#         Yo = np.stack([data.labels[u] for u in data.oof_ids]).astype(np.float32)
#         np.savez(os.path.join(cfg.HERE, f"knee_oof_{cfg.tag}_fold{cfg.fold}.npz"),
#                  pred=Po, truth=Yo, ids=np.array(data.oof_ids), fold=cfg.fold, nfolds=cfg.folds)
#         print(f"[cv] wrote OOF for best epoch ({len(data.oof_ids)} studies)", flush=True)

#     if len(run.topk) < 2:
#         return

#     sds = [t["model"] for t in run.topk]
#     avg = {}
#     for k in sds[0]:
#         v0 = sds[0][k]
#         avg[k] = (sum(sd[k].double() for sd in sds) / len(sds)).to(v0.dtype) if v0.is_floating_point() \
#             else v0.clone()
#     model.load_state_dict(avg)
#     au_swa, aucs_swa, P_swa = evaluate(model, vl, val_y, dev, desc="SWA eval")
#     eps = [int(t["epoch"]) for t in run.topk]
#     torch.save({**ckpt_meta(cfg, head_cfg, bb_src), "model": avg, "gold_auc": au_swa, "aucs": aucs_swa,
#                 "epoch": eps, "swa_over": eps}, os.path.join(cfg.HERE, f"knee_{cfg.tag}_swa.pt"))
#     np.savez(os.path.join(cfg.HERE, f"knee_gold_{cfg.tag}_swa.npz"),
#              pred=P_swa, truth=val_y, ids=np.array(run.gold_ids))
#     print(f"SWA over epochs {eps} | gold {au_swa:.4f} (best-epoch {run.best:.4f}) "
#           f"{'BETTER' if au_swa > run.best else 'no gain'}", flush=True)

#     Pens = np.mean([t["P"] for t in run.topk], axis=0)
#     ens, _ = macro_auc(Pens, val_y)
#     np.savez(os.path.join(cfg.HERE, f"knee_gold_{cfg.tag}_epens.npz"),
#              pred=Pens, truth=val_y, ids=np.array(run.gold_ids))
#     print(f"TOPK epochs {eps} | best {run.best:.4f} | epoch-ensemble {ens:.4f} "
#           f"({'BETTER' if ens > run.best else 'no gain'})", flush=True)
#     model.load_state_dict(run.best_state["model"])


# # ==============================================================================
# # 9. MAIN
# # ==============================================================================
# def seed_everything(seed):
#     random.seed(seed)
#     np.random.seed(seed)
#     torch.manual_seed(seed)
#     if torch.cuda.is_available():
#         torch.cuda.manual_seed_all(seed)


# def main():
#     cfg = apply_cli_overrides(Config())
#     seed_everything(cfg.seed)
#     dev = "cuda" if torch.cuda.is_available() else "cpu"
#     setup_gpus(cfg)
#     # Eval feeds a different number of windows per study; with benchmark=True cuDNN would autotune the whole
#     # network again for every new shape, which can stall the baseline / epoch-end evaluation for a very long time.
#     torch.backends.cudnn.benchmark = False
#     print(f"device {dev} | arch {cfg.arch} | res {cfg.res} | norm {cfg.norm} | "
#           f"aug {'ON' if cfg.aug else 'OFF'} | tag {cfg.tag}", flush=True)

#     labels_csv = cfg.labels_csv or prepare_labels(cfg)
#     ids = open_ids(cfg.corpus_root)
#     id2row = {uid: i for i, uid in enumerate(ids)}
#     print(f"[data] corpus ids: {len(ids)}", flush=True)
#     data = build_label_sets(cfg, ids, labels_csv)

#     model, head_cfg, bb_src = build_model(cfg, dev)
#     run = train(cfg, model, head_cfg, bb_src, data, id2row, dev)
#     post_training(cfg, model, head_cfg, bb_src, data, id2row, run, dev)


# if __name__ == "__main__":
#     main()